# Conventional augmentation experiment
Run on Kaggle with Skin Cancer ISIC, ISIC 2019 Training Ground Truth, and ISIC 2019 Metadata attached and a GPU enabled. The single code cell rebuilds the verified split and checks its frozen membership fingerprint before training.

The CNN architecture, class weights, learning rate, batch size, epoch budget and early-stopping rule match the recovered baseline. Training augmentation: horizontal/vertical flips, rotation up to 0.05 turns (18 degrees), zoom factor 0.10, reflection filling. Validation uses original resized images; test images are not evaluated.

The comparison is an initial single-run experiment. The same seed does not guarantee identical random trajectories or fully deterministic GPU results; repeated seeds are needed for a robust conclusion.

Keras documentation: https://keras.io/api/layers/preprocessing_layers/image_augmentation/random_rotation/ and https://keras.io/api/layers/preprocessing_layers/image_augmentation/random_zoom/


Import the split dataset and inspect

In [ ]:
#Import all necessary libraries
from pathlib import Path
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, balanced_accuracy_score
#Import all necessary libraries
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from hashlib import sha256
import json, gc, zipfile
import numpy as np
import tensorflow as tf
from sklearn.metrics import (classification_report, confusion_matrix,
    accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score,
    average_precision_score)

In [ ]:
MAPPING = {**dict.fromkeys(['NV','BKL','DF','VASC'], 'benign'),
           **dict.fromkeys(['MEL','BCC','SCC'], 'malignant')}

fingerprint_path = Path.cwd().parent / "artifacts" / "dataset_fingerprint.json"

with open(fingerprint_path) as f:
    fingerprint_data = json.load(f)
fingerprint = fingerprint_data["fingerprint"]

In [ ]:
data_path = Path.cwd().parent / "artifacts" / "isic2019_binary_split.csv"
out = Path.cwd().parent /'artifacts'
out.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(data_path)
print(df.head())
print(df.columns)
print(df["split"].value_counts())
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

       image_id                                               path   label  \
0  ISIC_0000000  /kaggle/input/datasets/nodoubttome/skin-cancer...  benign   
1  ISIC_0000006  /kaggle/input/datasets/nodoubttome/skin-cancer...  benign   
2  ISIC_0000007  /kaggle/input/datasets/nodoubttome/skin-cancer...  benign   
3  ISIC_0000009  /kaggle/input/datasets/nodoubttome/skin-cancer...  benign   
4  ISIC_0000011  /kaggle/input/datasets/nodoubttome/skin-cancer...  benign   

  lesion_id               group  split original_label  target  
0       NaN  image:ISIC_0000000  train             NV       0  
1       NaN  image:ISIC_0000006  train             NV       0  
2       NaN  image:ISIC_0000007  train             NV       0  
3       NaN  image:ISIC_0000009  train             NV       0  
4       NaN  image:ISIC_0000011  train             NV       0  
Index(['image_id', 'path', 'label', 'lesion_id', 'group', 'split',
       'original_label', 'target'],
      dtype='str')
split
train    972
test  

In [ ]:
train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"] == "val"].copy()
test_df  = df[df["split"] == "test"].copy()

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# Create a consistent mapping of class names to integer IDs
class_names = sorted(df["label"].unique().tolist())
class_to_id = {
    "benign": 0,
    "malignant": 1
}

assert set(df["label"].unique()) == set(class_to_id.keys())

class_names = ["benign", "malignant"]

def make_dataset(frame, training=False, seed=42):

    paths = frame["path"].astype(str).values
    labels = frame["label"].map(class_to_id).astype("float32").values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_image(
            image, channels=3, expand_animations=False
        )
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32) / 255.0

        return image, label

    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=seed,
            reshuffle_each_iteration=True
        )

    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

In [10]:
y_train = train_df["label"].map(class_to_id).to_numpy()
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, weights)
}

print(class_weights)
class_to_num = class_to_id
num_to_class = {v: k for k, v in class_to_num.items()}

{0: 0.9169811320754717, 1: 1.0995475113122173}


CNN

In [ ]:
def build_cnn():
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(128, 128, 3)),

        tf.keras.layers.Conv2D(
            32, 3, padding="same", activation="relu"
        ),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(
            64, 3, padding="same", activation="relu"
        ),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(
            128, 3, padding="same", activation="relu"
        ),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(
            256, 3, padding="same", activation="relu"
        ),

        tf.keras.layers.GlobalAveragePooling2D(),

        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.35),

        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    return model

In [ ]:
def build_augmentation(seed):

    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip(
            "horizontal_and_vertical", seed=seed
        ),

        tf.keras.layers.RandomRotation(
            0.05, fill_mode="reflect", seed=seed + 1
        ),

        tf.keras.layers.RandomZoom(
            0.10, fill_mode="reflect", seed=seed + 2
        ),

        tf.keras.layers.RandomTranslation(
            height_factor=0.10,
            width_factor=0.10,
            fill_mode="reflect",
            seed=seed + 3
        ),

        tf.keras.layers.RandomCrop(
            height=110, width=110, seed=seed + 4
        ),

        tf.keras.layers.Resizing(128, 128),

        tf.keras.layers.RandomBrightness(
            0.15, value_range=(0, 1), seed=seed + 5
        ),

        tf.keras.layers.RandomContrast(
            0.15, seed=seed + 6
        )

    ], name=f"conventional_augmentation_seed_{seed}")


In [ ]:
results = []

for seed in [42, 43, 44]:

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    # 1. Build CNN
    model = build_cnn()

    # 2. Load matching initial weights from baseline notebook
    weights_file = out / f"initial_weights_seed{seed}.npz"

    with np.load(weights_file) as weights:
        initial_weights = [
            weights[key]
            for key in sorted(
                weights.files,
                key=lambda k: int(k.split("_")[-1])
            )
        ]

    model.set_weights(initial_weights)

    init_hash = sha256(
        b"".join(w.tobytes() for w in model.get_weights())
    ).hexdigest()

    # 3. Create datasets
    train_ds = make_dataset(train_df, training=True, seed=seed)
    val_ds = make_dataset(val_df, training=False)

    # 4. Apply augmentation ONLY to training data
    augmentation = build_augmentation(seed)

    train_ds = train_ds.map(
        lambda images, labels: (
            tf.clip_by_value(
                augmentation(images, training=True),
                0.0,
                1.0
            ),
            labels
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    ).prefetch(tf.data.AUTOTUNE)

    # 5. Compile model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    run = f"binary_aug_seed{seed}"
    checkpoint = out / f"{run}.keras"

    # 6. Train model
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=25,
        class_weight=class_weights,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(
                str(checkpoint),
                monitor="val_loss",
                save_best_only=True
            ),
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True
            )
        ],
        verbose=2
    )

    # 7. Load best model
    best = tf.keras.models.load_model(checkpoint)

    # 8. Validation predictions
    probability = best.predict(val_ds, verbose=0).reshape(-1)

    y = val_df["label"].map(class_to_id).to_numpy()
    pred = (probability >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    # 9. Calculate metrics
    row = dict(
        seed=seed,
        condition="binary_aug",
        epochs_trained=len(history.history["loss"]),
        selected_epoch=int(
            np.argmin(history.history["val_loss"]) + 1
        ),
        validation_loss=float(min(history.history["val_loss"])),
        accuracy=float(accuracy_score(y, pred)),
        macro_f1=float(f1_score(
            y, pred, labels=[0, 1],
            average="macro", zero_division=0
        )),
        balanced_accuracy=float(balanced_accuracy_score(y, pred)),
        malignant_sensitivity=float(tp / (tp + fn)),
        specificity=float(tn / (tn + fp)),
        roc_auc=float(roc_auc_score(y, probability)),
        average_precision=float(average_precision_score(y, probability)),
        threshold=0.5,
        initial_weights_sha256=init_hash,
        split_fingerprint=fingerprint,
        test_evaluated=False
    )

    report = classification_report(
        y,
        pred,
        labels=[0, 1],
        target_names=["benign", "malignant"],
        output_dict=True,
        zero_division=0
    )

    # 10. Save metrics
    (out / f"{run}_metrics.json").write_text(
        json.dumps(
            dict(**row, classification_report=report),
            indent=2
        )
    )

    # 11. Save training history
    pd.DataFrame(history.history).assign(
        epoch=lambda d: np.arange(1, len(d) + 1)
    ).to_csv(
        out / f"{run}_history.csv",
        index=False
    )

    # 12. Save validation predictions
    predictions = val_df[
        ["image_id", "original_label", "label", "group"]
    ].copy()

    predictions["malignant_probability"] = probability
    predictions["prediction"] = pred

    predictions.to_csv(
        out / f"{run}_validation_predictions.csv",
        index=False
    )

    # 13. Save confusion matrix
    pd.DataFrame(
        confusion_matrix(y, pred, labels=[0, 1]),
        index=["benign", "malignant"],
        columns=["benign", "malignant"]
    ).to_csv(
        out / f"{run}_confusion_matrix.csv"
    )

    # 14. Save results so far
    results.append(row)

    pd.DataFrame(results).to_csv(
        out / "binary_aug_results.csv",
        index=False
    )

    # 15. Archive this seed's artifacts
    archive_path = out / f"{run}_artifacts.zip"

with zipfile.ZipFile(
    archive_path, "w", zipfile.ZIP_DEFLATED
) as archive:

    for p in out.glob(f"{run}*"):
        if p.is_file() and p != archive_path:
            archive.write(p, p.name)

    archive.write(weights_file, weights_file.name)
    archive.write(data_path, data_path.name)

print("Archive saved:", archive_path)
print("AUGMENTED_RUN_COMPLETE", json.dumps(row), flush=True)

del model, best, train_ds, val_ds
gc.collect()